In [0]:
%sql
-- ===========================================
-- Quantidade de avaliações que o vendedor recebeu 
-- ===========================================
SELECT
    oi.seller_id,
    COUNT(DISTINCT o.order_id) AS total_reviews
FROM (
    SELECT DISTINCT seller_id, order_id
    FROM olist.order_items
) oi
JOIN olist.orders o 
    ON oi.order_id = o.order_id
JOIN olist.order_reviews r 
    ON o.order_id = r.order_id
GROUP BY oi.seller_id
ORDER BY total_reviews DESC;

In [0]:
%sql
-- ===========================================
-- Percentual de avaliações recebidas pelo vendedor*: Quantidade de avaliações recebidas / Quantidade de pedidos recebidos
-- ===========================================
WITH params AS (
    SELECT
        CAST('2018-07-02' AS DATE) AS data_ref   -- << Data de referencia
),

-- ===========================================
-- RELAÇÃO ÚNICA SELLER x ORDER
-- ===========================================
seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        o.order_id,
        CAST(o.order_purchase_timestamp AS DATE) AS order_date
    FROM olist.order_items oi
    JOIN olist.orders o
        ON oi.order_id = o.order_id
),

-- ===========================================
-- FLAG DE REVIEW
-- ===========================================
orders_flag AS (
    SELECT
        so.order_id,
        so.order_date,
        CASE 
            WHEN r.order_id IS NOT NULL THEN 1
            ELSE 0
        END AS has_review
    FROM seller_orders so
    LEFT JOIN (
        SELECT DISTINCT order_id
        FROM olist.order_reviews
    ) r
        ON so.order_id = r.order_id
)

-- ===========================================
-- RESULTADO FINAL
-- ===========================================
SELECT

    -- ======================
    -- 28 DIAS
    -- ======================
    COUNT(DISTINCT CASE
        WHEN f.order_date BETWEEN date_sub(p.data_ref, 27) AND p.data_ref
        THEN f.order_id
    END) AS pedidos_28d,

    COUNT(DISTINCT CASE
        WHEN f.order_date BETWEEN date_sub(p.data_ref, 27) AND p.data_ref
             AND f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_28d,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.order_date BETWEEN date_sub(p.data_ref, 27) AND p.data_ref
                 AND f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT CASE
            WHEN f.order_date BETWEEN date_sub(p.data_ref, 27) AND p.data_ref
            THEN f.order_id
        END), 0)
    , 2) AS percentual_28d,

    -- ======================
    -- 56 DIAS
    -- ======================
    COUNT(DISTINCT CASE
        WHEN f.order_date BETWEEN date_sub(p.data_ref, 55) AND p.data_ref
        THEN f.order_id
    END) AS pedidos_56d,

    COUNT(DISTINCT CASE
        WHEN f.order_date BETWEEN date_sub(p.data_ref, 55) AND p.data_ref
             AND f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_56d,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.order_date BETWEEN date_sub(p.data_ref, 55) AND p.data_ref
                 AND f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT CASE
            WHEN f.order_date BETWEEN date_sub(p.data_ref, 55) AND p.data_ref
            THEN f.order_id
        END), 0)
    , 2) AS percentual_56d,

    -- ======================
    -- 365 DIAS
    -- ======================
    COUNT(DISTINCT CASE
        WHEN f.order_date BETWEEN date_sub(p.data_ref, 364) AND p.data_ref
        THEN f.order_id
    END) AS pedidos_365d,

    COUNT(DISTINCT CASE
        WHEN f.order_date BETWEEN date_sub(p.data_ref, 364) AND p.data_ref
             AND f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_365d,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.order_date BETWEEN date_sub(p.data_ref, 364) AND p.data_ref
                 AND f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT CASE
            WHEN f.order_date BETWEEN date_sub(p.data_ref, 364) AND p.data_ref
            THEN f.order_id
        END), 0)
    , 2) AS percentual_365d,

    -- ======================
    -- LIFETIME (TODA VIDA)
    -- ======================
    COUNT(DISTINCT f.order_id) AS pedidos_lifetime,

    COUNT(DISTINCT CASE
        WHEN f.has_review = 1
        THEN f.order_id
    END) AS avaliacoes_lifetime,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN f.has_review = 1
            THEN f.order_id
        END) * 100.0
        /
        NULLIF(COUNT(DISTINCT f.order_id), 0)
    , 2) AS percentual_lifetime

FROM orders_flag f
CROSS JOIN params p;